In [1]:
import json
from pathlib import Path
import os
from databases import quest_db, sqlite_db
import pandas as pd
# from data import HoldingsTimelineCalculator  # Adjust import path


import numpy as np
from datetime import datetime, timedelta, date
from typing import Dict, List, Optional, Literal

import requests
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

import sqlite3


In [16]:
def fetch_sql_db(query: str, db_path: str="../src_django/db.sqlite3") -> pd.DataFrame:
    """Fetch data from a SQLite database and return it as a DataFrame."""
    try:
        with sqlite3.connect(db_path) as conn:
            df = pd.read_sql_query(query, conn)
            return df
    except Exception as e:
        print(f"Error executing query: {e}")
        return pd.DataFrame()  # Return an empty DataFrame on error
    finally:        
        conn.close()


query = f"""WITH base AS (
            SELECT 
                tradingsymbol AS symbol,

                SUM(
                    CASE 
                        WHEN transaction_type = 'BUY'  THEN quantity
                        WHEN transaction_type = 'SELL' THEN -quantity
                        ELSE 0 
                    END
                ) AS net_quantity,

                SUM(
                    CASE 
                        WHEN transaction_type = 'BUY'
                        THEN quantity * average_price
                        ELSE 0 
                    END
                ) AS total_buy_value,

                SUM(
                    CASE 
                        WHEN transaction_type = 'BUY'
                        THEN quantity
                        ELSE 0
                    END
                ) AS total_buy_qty

            FROM users_auth_trades
            GROUP BY tradingsymbol
        )

        SELECT
            symbol,
            net_quantity,

            total_buy_value / NULLIF(total_buy_qty, 0) AS avg_price,

            total_buy_value AS total_capital_employed,

            net_quantity * (total_buy_value / NULLIF(total_buy_qty, 0)) AS current_invested

        FROM base;
    """

stock_holding = fetch_sql_db(query)
# stock_holding.head(50)

symbols = stock_holding["symbol"].unique().tolist()

def fetch_latest_prices(symbols):
    sym_list = ",".join(f"'{s}'" for s in symbols)

    sql = f"""
            WITH latest_price AS (
                SELECT
                    CAST(security_id AS INT) AS sid,
                    close
                FROM daily_historical_prices
                LATEST ON "timestamp" PARTITION BY CAST(security_id AS INT)
            )
            SELECT
                d.underlying_symbol AS symbol,
                lp.close AS current_price
            FROM latest_price lp
            JOIN dhan_full_instruments_list d
                ON lp.sid = CAST(d.security_id AS INT)
            WHERE d.underlying_symbol IN ({sym_list})
            AND d.exch_id = 'NSE'
            AND d.segment = 'E';
    """
    return quest_db.read_questdb_dataframe(query=sql)

latest_prices = fetch_latest_prices(symbols)
latest_prices.head(50)

# def fetch_security_id(exchange='NSE', segment='E', symbol='COALINDIA'):
#     daily_price_sql1 = f"""
#                 SELECT security_id
#                 FROM dhan_full_instruments_list
#                 WHERE EXCH_ID = '{exchange}' AND SEGMENT = '{segment}' AND UNDERLYING_SYMBOL='{symbol}';
#                 """
#     security_id = quest_db.read_questdb_dataframe(query = daily_price_sql1).values[0][0]
#     print(f"security_id: {security_id}")
#     return security_id

# security_id = fetch_security_id(exchange='NSE', segment='E', symbol='COALINDIA')

# daily_price_sql1 = f"""
#                 SELECT CLOSE
#                 FROM daily_historical_prices
#                 WHERE SECURITY_ID = '{security_id}' AND timestamp = (SELECT MAX(timestamp) FROM daily_historical_prices WHERE security_id = '{security_id}' );
#                 """
# print(daily_price_sql1)
# current_price = quest_db.read_questdb_dataframe(query = daily_price_sql1)

# print(current_price)

# markdown = f"""
#     ## {data['symbol']} — Portfolio Holding

#     | Metric | Value |
#     |---|---|
#     | **Ticker** | {data['symbol']} |
#     | **Shares Held** | {data['net_quantity']:,} |
#     | **Avg. Buy Price** | ${data['avg_price']:.2f} |
#     | **Total Invested** | ${data['total_capital_employed']:,.2f} |
#     | **Current Price** | ${data['current_price']:.2f} |
#     | **Current Value** | ${data['current_value']:,.2f} | 
#     | **Realized P&L** | ${data['realized_pnl']:,.2f} ({data['realized_pct']:.2f}%) |
#     | **Unrealized P&L** | ${data['unrealized_pnl']:,.2f} ({data['pnl_pct']:.2f}%) |
#     | **% of Portfolio** | {data['portfolio_weight']:.2f}% |
# """


QuestDB returned an error: table and column names that are SQL keywords have to be enclosed in double quotes, such as "CAST"


,error
0,table and column names that are SQL keywords h...


In [5]:
daily_price_sql = f"""
            select *
            from pnl
            where symbol='INFY'
            """

pnl = quest_db.read_questdb_dataframe(query= daily_price_sql)
pnl.head()
print(pnl.head().iloc[0, 1])

dff = pnl[pnl["symbol"] == 'INFY'].copy()

dff["year"] = pd.to_datetime(dff["timestamp"]).dt.year

dff["items"] = dff["items"].apply(lambda x: json.loads(x) if isinstance(x, str) else x)

print(dff.head())

records = []
for _, row in dff.iterrows():
    records.append({
        "year": int(row["year"]),
        **row["items"]
    })

print(records)


{"Sales": 62441, "Raw Material Cost": null, "Change in Inventory": null, "Power and Fuel": 217, "Other Mfr. Exp": 5860, "Employee Cost": 34415, "Selling and admin": 4441, "Other Expenses": 429, "Other Income": 3120, "Depreciation": 1459, "Interest": null, "Profit before tax": 18740, "Tax": 5251, "Net profit": 13489, "Dividend Amount": 5548.4}
  symbol                                              items  \
0   INFY  {'Sales': 62441, 'Raw Material Cost': None, 'C...   
1   INFY  {'Sales': 68484, 'Raw Material Cost': None, 'C...   
2   INFY  {'Sales': 70522, 'Raw Material Cost': None, 'C...   
3   INFY  {'Sales': 82675, 'Raw Material Cost': None, 'C...   
4   INFY  {'Sales': 90791, 'Raw Material Cost': None, 'C...   

                     timestamp  year  
0  2016-03-31T00:00:00.000000Z  2016  
1  2017-03-31T00:00:00.000000Z  2017  
2  2018-03-31T00:00:00.000000Z  2018  
3  2019-03-31T00:00:00.000000Z  2019  
4  2020-03-31T00:00:00.000000Z  2020  
[{'year': 2016, 'Sales': 62441, 'Raw Mater

In [2]:
def fetch_ohlc_data(symbol, exchange='NSE', interval='1d', start='1975-01-01', end='2025-12-31', segment='E'):
        daily_price_sql1 = f"""
                    SELECT SECURITY_ID, UNDERLYING_SYMBOL, SYMBOL_NAME, EXCH_ID
                    FROM dhan_full_instruments_list
                    WHERE EXCH_ID = '{exchange}' AND UNDERLYING_SYMBOL = '{symbol}'  AND SEGMENT = '{segment}';
                    """
        security_id = quest_db.execute_query(sql_query = daily_price_sql1)['dataset'][0][0]
        print(f"security_id: {security_id}")
        daily_price_sql2 = f"""
                    SELECT timestamp as date, open, high, low, close, volume, '{symbol}' AS symbol, '{exchange}' AS exchange
                    FROM daily_historical_prices 
                    WHERE security_id = {security_id} AND timestamp >= '{start}' AND timestamp <= '{end}'; 
                    """ 

        return quest_db.read_questdb_dataframe(query= daily_price_sql2)


def resolve_dates(range: str, start_date=None, end_date=None):
    today = pd.Timestamp.today().normalize()

    range = range.upper()

    if range == "1M":
        return today - pd.DateOffset(months=1), today
    if range == "3M":
        return today - pd.DateOffset(months=3), today
    if range == "6M":
        return today - pd.DateOffset(months=6), today
    if range == "YTD":
        return pd.Timestamp(today.year, 1, 1), today
    if range == "1Y":
        return today - pd.DateOffset(years=1), today
    if range == "3Y":
        return today - pd.DateOffset(years=3), today
    if range == "5Y":
        return today - pd.DateOffset(years=5), today
    if range == "LTD":
        return pd.Timestamp("2000-01-01"), today

    raise ValueError(f"Invalid range: {range}")

def ohlc_resample(df: pd.DataFrame):
    span_days = (df['date'].max() - df['date'].min()).days

    if span_days <= 400:
        return df

    if span_days <= 2500:
        rule = "W"

    else:
        rule = "M"

    ohlc = {
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum"
    }

    return df.set_index("date").resample(rule).agg(ohlc).dropna().reset_index()


In [7]:
start, end = resolve_dates('5Y')
df = fetch_ohlc_data('AXISBANK', start=start, end=end)
df['date'] = pd.to_datetime(df['date'], utc=False)
df2 = ohlc_resample(df)

df2.head()

security_id: 5900


,date,open,high,low,close,volume
0,2021-02-21 00:00:00+00:00,775.00,782.0,715.00,719.45,53443652.0
1,2021-02-28 00:00:00+00:00,732.35,783.5,712.60,728.55,100254532.0
2,2021-03-07 00:00:00+00:00,730.45,760.8,720.50,743.30,75260999.0
3,2021-03-14 00:00:00+00:00,758.70,776.6,716.65,744.40,71139559.0
4,2021-03-21 00:00:00+00:00,751.00,752.0,703.50,716.20,71180996.0


In [ ]:
df = fetch_ohlc_data(symbol="RELIANCE")
df.head()



fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['date'],
    y=df['close'],
    mode='lines',
    line=dict(width=2, color='#1A73E8'), # Google Blue
    name="RELIANCE",
    fill='tozeroy', # Optional: gives that subtle Google area-chart look
    fillcolor='rgba(26, 115, 232, 0.05)'
))

# Google Finance style range selection logic
fig.update_layout(
    template="plotly_white",
    dragmode="select",  # This enables the "drag to highlight" behavior
    hovermode="x unified",
    clickmode="event+select",
    xaxis=dict(
        type="date",
        showspikes=True,
        spikemode="across",
        spikesnap="data",
        showgrid=False,
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(count=5, label="5Y", step="year", stepmode="backward"),
                dict(label="LTD", step="all")
            ])
        )
    ),
    yaxis=dict(side="right", showgrid=True, gridcolor="#f0f0f0"),
    # Selectbox styling to make the drag-area look like a measurement tool
    selectdirection="h", 
    activeselection=dict(fillcolor="rgba(26, 115, 232, 0.2)", opacity=0.5)
)

def handle_selection(trace, points, selector):

    if not selector or not selector.xrange:
        return

    x0, x1 = selector.xrange
    mask = (df['date'] >= x0) & (df['date'] <= x1)
    sel = df.loc[mask]

    if len(sel) < 2:
        return

    p0 = sel['close'].iloc[0]
    p1 = sel['close'].iloc[-1]

    delta = p1 - p0
    pct = (delta / p0) * 100

    annotation_text = (
        f"<b>{x0.date()} → {x1.date()}</b><br>"
        f"Δ ₹{delta:.2f} ({pct:.2f}%)"
    )

    with fig.batch_update():
        fig.layout.shapes = [
            dict(
                type="rect",
                xref="x",
                yref="paper",
                x0=x0,
                x1=x1,
                y0=0,
                y1=1,
                fillcolor="rgba(26,115,232,0.15)",
                line_width=0
            )
        ]

        fig.layout.annotations = [
            dict(
                x=x1,
                y=1.05,
                xref="x",
                yref="paper",
                text=annotation_text,
                showarrow=False,
                bgcolor="rgba(255,255,255,0.9)",
                bordercolor="#1A73E8",
                borderwidth=1
            )
        ]

# Attach callback
fig.data[0].on_selection(handle_selection)

fig.show()

security_id: 2885


In [3]:
params = {
    "query": "sbi",   # Required, length >= 1
    "exchange": "NSE", # Optional
    "limit": 10        # Optional, between 1 and 100
}
url = 'http://127.0.0.1:6000/stock/search'

response = requests.get(url, params=params)

if response.status_code == 422:
    print("Error Details:", response.json()) # FastAPI tells you EXACTLY what failed
else:
    print("Success:", json.dumps(response.json(), indent=2))

Success: [
  {
    "label": "SFMP6DD - SBIAMC - SFMP6DD",
    "value": "SFMP6DD",
    "extraInfo": {
      "description": "all",
      "rightOfDescription": "NSE"
    }
  },
  {
    "label": "SFMP6DR - SBIAMC - SFMP6DR",
    "value": "SFMP6DR",
    "extraInfo": {
      "description": "all",
      "rightOfDescription": "NSE"
    }
  },
  {
    "label": "SFMP6GD - SBIAMC - SFMP6GD",
    "value": "SFMP6GD",
    "extraInfo": {
      "description": "all",
      "rightOfDescription": "NSE"
    }
  },
  {
    "label": "SFMP6GR - SBIAMC - SFMP6GR",
    "value": "SFMP6GR",
    "extraInfo": {
      "description": "all",
      "rightOfDescription": "NSE"
    }
  },
  {
    "label": "SETFNIF50 - SBI-ETF NIFTY 50",
    "value": "SETFNIF50",
    "extraInfo": {
      "description": "all",
      "rightOfDescription": "NSE"
    }
  },
  {
    "label": "SFMP72DD - SBIAMC - SFMP72DD",
    "value": "SFMP72DD",
    "extraInfo": {
      "description": "all",
      "rightOfDescription": "NSE"
    }
  },
  {


In [12]:
content=json.load((Path(os.path.join(os.getcwd())).resolve() / "apps.json").open())

In [2]:

daily_price_sql = f"""
            SELECT table_name FROM tables();
            """

acc_daily_prices = quest_db.read_questdb_dataframe(query= daily_price_sql)
acc_daily_prices.head()

,table_name
0,dhan_api_fetch_errors
1,daily_historical_prices
2,cf
3,nse_eq_metadata
4,dhan_full_instruments_list


In [56]:
def fetch_equities_list(exchange='NSE', segment='E'):
    daily_price_sql1 = f"""
                SELECT UNDERLYING_SYMBOL as symbol, SYMBOL_NAME as name, 'all' as sector, EXCH_ID as exchange
                FROM dhan_full_instruments_list
                WHERE EXCH_ID = '{exchange}' AND SEGMENT = '{segment}';
                """
    # security_id = quest_db.execute_query(sql_query = daily_price_sql1)['dataset'][0][0]
    # print(f"security_id: {security_id}")
    # daily_price_sql2 = f"""
    #             SELECT timestamp as date, open, high, low, close, volume, '{symbol}' AS symbol, '{exchange}' AS exchange
    #             FROM daily_historical_prices 
    #             WHERE security_id = {security_id} AND timestamp >= '{start}' AND timestamp <= '{end}'; 
    #             """ 

    return quest_db.read_questdb_dataframe(query= daily_price_sql1)

acc_daily_prices = fetch_equities_list(exchange='NSE', segment='E')
stocks_db = acc_daily_prices[['symbol', 'name', 'sector', 'exchange']].to_dict('records')

stocks_db[:5]


[{'symbol': 'GOLDSTAR',
  'name': 'GOLDSTAR POWER LIMITED',
  'sector': 'all',
  'exchange': 'NSE'},
 {'symbol': 'ARE&M',
  'name': 'AMARA RAJA ENERGY MOB LTD',
  'sector': 'all',
  'exchange': 'NSE'},
 {'symbol': '656MH32',
  'name': 'SDL MH 6.56% 2032',
  'sector': 'all',
  'exchange': 'NSE'},
 {'symbol': '94SFL28',
  'name': 'SEC RED NCD 9.40% SR. V',
  'sector': 'all',
  'exchange': 'NSE'},
 {'symbol': '679AP34',
  'name': 'SDL AP 6.79% 2034',
  'sector': 'all',
  'exchange': 'NSE'}]

In [20]:
def fetch_ohcl_data(symbol, exchange='NSE', interval='1d', start='2025-01-01', end='2025-12-31', segment='E'):
    daily_price_sql1 = f"""
                SELECT SECURITY_ID, UNDERLYING_SYMBOL, SYMBOL_NAME, EXCH_ID
                FROM dhan_full_instruments_list
                WHERE EXCH_ID = '{exchange}' AND UNDERLYING_SYMBOL = '{symbol}'  AND SEGMENT = '{segment}';
                """
    security_id = quest_db.execute_query(sql_query = daily_price_sql1)['dataset'][0][0]
    print(f"security_id: {security_id}")
    daily_price_sql2 = f"""
                SELECT timestamp as date, open, high, low, close, volume, '{symbol}' AS symbol, '{exchange}' AS exchange
                FROM daily_historical_prices 
                WHERE security_id = {security_id} AND timestamp >= '{start}' AND timestamp <= '{end}'; 
                """ 

    return quest_db.read_questdb_dataframe(query= daily_price_sql2)

acc_daily_prices = fetch_ohcl_data(symbol='ARE&M', exchange='NSE', start='2025-01-01', end='2025-12-31')
print(acc_daily_prices.head())

security_id: 100
Empty DataFrame
Columns: [date, open, high, low, close, volume, symbol, exchange]
Index: []


In [5]:
daily_price_sql = f"""
            select *
            from dhan_api_fetch_errors
            LIMIT 10
            """

acc_daily_prices = quest_db.read_questdb_dataframe(query= daily_price_sql)
acc_daily_prices.head()

,security_id,exchange_segment,error_response,timestamp
0,505032,BSE_EQ,"{""status"": ""failure"", ""remarks"": {""error_code""...",2026-01-01T17:12:11.537738000Z
1,506024,BSE_EQ,"{""status"": ""failure"", ""remarks"": {""error_code""...",2026-01-01T17:12:11.537738000Z
2,506162,BSE_EQ,"{""status"": ""failure"", ""remarks"": {""error_code""...",2026-01-01T17:12:11.537738000Z
3,506178,BSE_EQ,"{""status"": ""failure"", ""remarks"": {""error_code""...",2026-01-01T17:12:11.537738000Z
4,506580,BSE_EQ,"{""status"": ""failure"", ""remarks"": {""error_code""...",2026-01-01T17:12:11.537738000Z


In [6]:
daily_price_sql = f"""
            select *
            from cf
            LIMIT 10
            """

acc_daily_prices = quest_db.read_questdb_dataframe(query= daily_price_sql)
acc_daily_prices.head()

,symbol,items,timestamp
0,SHRYDUS,"{""Cash from Operating Activity"": null, ""Cash f...",2003-03-31T00:00:00.000000Z
1,SHRYDUS,"{""Cash from Operating Activity"": null, ""Cash f...",2004-03-31T00:00:00.000000Z
2,AIRFLOA,"{""Cash from Operating Activity"": null, ""Cash f...",2005-03-31T00:00:00.000000Z
3,SHRYDUS,"{""Cash from Operating Activity"": null, ""Cash f...",2005-03-31T00:00:00.000000Z
4,CRSL,"{""Cash from Operating Activity"": null, ""Cash f...",2005-03-31T00:00:00.000000Z


In [7]:
daily_price_sql = f"""
            select *
            from nse_eq_metadata
            LIMIT 10
            """

acc_daily_prices = quest_db.read_questdb_dataframe(query= daily_price_sql)
acc_daily_prices.head()

,Symbol,Data_JSON,timestamp
0,CUMMINSIND,"{""info"": {""symbol"": ""CUMMINSIND"", ""companyName...",2025-11-29T06:22:23.035226000Z
1,GS150326C-GS,"{""error"": {}, ""message"": ""TypeError: Cannot re...",2025-11-29T06:22:23.035226000Z
2,IIFLZC28-NG,"{""error"": {}, ""message"": ""TypeError: Cannot re...",2025-11-29T06:22:23.035226000Z
3,823SFL34-N0,"{""error"": {}, ""message"": ""TypeError: Cannot re...",2025-11-29T06:22:23.035226000Z
4,KODYTECH-SM,"{""error"": {}, ""message"": ""TypeError: Cannot re...",2025-11-29T06:22:23.035226000Z


In [6]:
daily_price_sql = f"""
            select *
            from dhan_full_instruments_list
            where segment='E'
            LIMIT 10
            """

acc_daily_prices = quest_db.read_questdb_dataframe(query= daily_price_sql)
acc_daily_prices.head()

,EXCH_ID,SEGMENT,SECURITY_ID,ISIN,INSTRUMENT,UNDERLYING_SECURITY_ID,UNDERLYING_SYMBOL,SYMBOL_NAME,DISPLAY_NAME,INSTRUMENT_TYPE,...,SELL_BO_MIN_MARGIN_PER,BUY_BO_SL_RANGE_MAX_PERC,SELL_BO_SL_RANGE_MAX_PERC,BUY_BO_SL_RANGE_MIN_PERC,SELL_BO_SL_MIN_RANGE,BUY_BO_PROFIT_RANGE_MAX_PERC,SELL_BO_PROFIT_RANGE_MAX_PERC,BUY_BO_PROFIT_RANGE_MIN_PERC,SELL_BO_PROFIT_RANGE_MIN_PERC,MTF_LEVERAGE
0,BSE,E,200072,INE813V01022,EQUITY,None,MCL,MADHAV COPPER LIMITED,Madhav Copper,ES,...,0,0,0,0,0,0,0,0,0,0
1,BSE,E,200105,INE786W01010,EQUITY,None,RKEC,RKEC PROJECTS LIMITED,RKEC Projects,ES,...,0,0,0,0,0,0,0,0,0,0
2,BSE,E,200184,INE749Y01014,EQUITY,None,AMJUMBO,A AND M JUMBO BAGS LTD,A & M Jumbo Bags,ES,...,0,0,0,0,0,0,0,0,0,0
3,BSE,E,200223,INE702Y01013,EQUITY,None,SMVD,SMVD POLY PACK LIMITED,SMVD Poly Pack,ES,...,0,0,0,0,0,0,0,0,0,0
4,BSE,E,200283,INE412C01023,EQUITY,None,JMA,JULLUNDUR MOT AGENCY LTD,Jullundur Motor Agency,ES,...,0,0,0,0,0,0,0,0,0,2


In [13]:
content

[{'name': 'Hello World',
  'img': '',
  'img_dark': '',
  'img_light': '',
  'description': 'Hello World template',
  'allowCustomization': True,
  'tabs': {'': {'id': '',
    'name': '',
    'layout': [{'i': 'hello_world',
      'x': 0,
      'y': 0,
      'w': 12,
      'h': 4,
      'state': {'params': {'name': ''}}}]}},
  'groups': []}]

In [14]:


class HoldingsTimelineCalculator:
    """
    Calculates portfolio holdings timeline from trades data.
    
    Aggregates trades by symbol (across exchanges) and tracks:
    - Quantity held over time
    - Weighted average price
    - Invested value
    - Daily holdings with carry-forward logic
    """
    
    def __init__(self, trades_df: pd.DataFrame):
        """
        Initialize calculator with trades dataframe.
        
        Args:
            trades_df: DataFrame with columns:
                - fill_timestamp: datetime of trade execution
                - tradingsymbol: symbol name
                - quantity: number of shares
                - average_price: price per share
                - transaction_type: 'BUY'/'buy' or 'SELL'/'sell'
                - account_id: account identifier
        """
        self.trades_df = trades_df.copy()
        self._prepare_data()
        self._fetch_portfolio_market_prices()
        self.price_map = {}
        self.sec_id_dict = {}
    
    def _prepare_data(self):
        """Prepare and clean the trades data."""
        # print(f"_prepare_data() in progress")
        # time1 = datetime.now()
        # Convert fill_timestamp to datetime
        self.trades_df['fill_timestamp'] = pd.to_datetime(
            self.trades_df['fill_timestamp']
        )
        
        # Extract date only (remove time component)
        self.trades_df['trade_date'] = self.trades_df['fill_timestamp'].dt.date
        
        # Standardize transaction type to uppercase
        self.trades_df['transaction_type'] = (
            self.trades_df['transaction_type'].str.upper()
        )
        
        # Handle tags - default to 'untagged' if null/empty
        if 'tag' not in self.trades_df.columns:
            self.trades_df['tag'] = 'untagged'
        else:
            self.trades_df['tag'] = self.trades_df['tag'].fillna('untagged')
            self.trades_df['tag'] = self.trades_df['tag'].replace('', 'untagged')
        
        # Sort by date and timestamp
        self.trades_df = self.trades_df.sort_values(
            ['trade_date', 'fill_timestamp']
        ).reset_index(drop=True)
        # print(f"Time taken: {datetime.now() - time1 }")
    
    def _fetch_portfolio_market_prices(self):
        # print(f"_fetch_portfolio_market_prices() in progress")
        # time2 = datetime.now()

        accounts = self.trades_df['account_id'].unique()

        self.market_prices = {}

        def get_all_market_prices(acc):

            account_trades = self.trades_df[
                self.trades_df['account_id'] == acc
            ].copy()
            
            if account_trades.empty:
                return self._empty_response(acc, granularity = 'daily')
            

            account_trades['fill_timestamp'] = pd.to_datetime(account_trades['fill_timestamp'])
            first_trade_date = (account_trades['fill_timestamp'].dt.date).min()

            
            daily_price_sql = f"""
            select *,
            date_trunc('day', timestamp) as ts_day
            from daily_historical_prices
            where 1=1
            AND security_id in ({str(list(account_trades['exchange_token'].unique()))[1:-1]})
            AND timestamp >= to_timestamp('{str(first_trade_date)}', 'yyyy-MM-dd')
            order by security_id, timestamp
            """

            acc_daily_prices = quest_db.read_questdb_dataframe(query= daily_price_sql)

            acc_daily_prices.loc[:, 'date'] = pd.to_datetime(acc_daily_prices['ts_day']).dt.date


            token_symbol_map = self.trades_df.groupby(['tradingsymbol', 'exchange_token', 'exchange'])['id'].count().reset_index().drop(columns=['id'])
            acc_daily_prices = acc_daily_prices.merge(token_symbol_map,
                            left_on='security_id',
                            right_on='exchange_token',
                            how='left')
            # print(f"Time taken: {datetime.now() - time2 }")
            return acc_daily_prices

        for acc in accounts:
            self.market_prices[acc] = get_all_market_prices(acc)
            pass
        # print(f"Time taken: {datetime.now() - time2 }")
        return self.market_prices
    
    def calculate_timeline(
        self,
        account_id: str,
        start_date: Optional[str] = None,
        end_date: Optional[str] = None,
        granularity: Literal['daily', 'trade_dates'] = 'daily'
    ) -> Dict:
        """
        Calculate holdings timeline for a specific account.
        
        Args:
            account_id: Account to calculate holdings for
            start_date: Start date (YYYY-MM-DD), defaults to first trade
            end_date: End date (YYYY-MM-DD), defaults to today
            granularity: 'daily' for all days, 'trade_dates' for trade days only
        
        Returns:
            Dictionary with holdings timeline in the specified format
        """
        # print(f"calculate_timeline() in progress")
        # time3 = datetime.now()

        # Filter trades for this account
        account_trades = self.trades_df[
            self.trades_df['account_id'] == account_id
        ].copy()
        
        if account_trades.empty:
            return self._empty_response(account_id, granularity)
        
        # Filter market prices for this account --> dataframe
        account_market_prices = self.market_prices[account_id]

        self.price_map = {
            (row.tradingsymbol, row.exchange_token, row.date): row.close
            for row in account_market_prices.itertuples(index=False)
        }
        self.sec_id_dict = (
            account_market_prices
            .drop_duplicates('tradingsymbol')
            .set_index('tradingsymbol')['exchange_token']
            .to_dict()
        )

        # Determine date range
        first_trade_date = account_trades['trade_date'].min()
        last_trade_date = account_trades['trade_date'].max()
        
        if start_date:
            start_date = pd.to_datetime(start_date).date()
            start_date = max(start_date, first_trade_date)
        else:
            start_date = first_trade_date
        
        if end_date:
            end_date = pd.to_datetime(end_date).date()
        else:
            end_date = datetime.now().date()
        
        # Filter trades within date range
        account_trades = account_trades[
            (account_trades['trade_date'] >= start_date) &
            (account_trades['trade_date'] <= end_date)
        ]
        
        # Calculate holdings based on granularity
        if granularity == 'trade_dates':
            timeline = self._calculate_trade_dates_timeline(
                account_trades, start_date, end_date
            )
        else:  # daily
            timeline = self._calculate_daily_timeline(
                account_trades, start_date, end_date
            )
        
        # Count trading days (days with actual trades)
        trading_days = len(account_trades['trade_date'].unique())
        total_days = (end_date - start_date).days + 1

        # print(f"Time taken: {datetime.now() - time3 }")

        return {
            'account_id': account_id,
            'currency': 'INR',
            'granularity': granularity,
            'date_range': {
                'start_date': start_date.isoformat(),
                'end_date': end_date.isoformat(),
                'total_days': total_days,
                'trading_days': trading_days
            },
            'timeline': timeline
        }

    def _calculate_trade_dates_timeline(
        self,
        trades: pd.DataFrame,
        start_date,
        end_date
    ) -> List[Dict]:

        # print(f"_calculate_trade_dates_timeline() in progress")
        # time4 = datetime.now()

        """Calculate holdings only for dates with trades."""
        timeline = []
        holdings_state = {}  # symbol -> bucket -> {qty, avg_price, invested_value}
        
        # Group trades by date
        for trade_date in trades['trade_date'].unique():
            date_trades = trades[trades['trade_date'] == trade_date]
            
            # Process all trades for this date
            trades_count_today = self._process_trades_for_date(
                date_trades, holdings_state
            )
            
            # Create snapshot for this date
            timeline_entry = self._create_timeline_entry(
                trade_date, holdings_state, trades_count_today
            )
            timeline.append(timeline_entry)
        # print(f"Time taken: {datetime.now() - time4 }")
        return timeline
    
    def _calculate_daily_timeline(
        self,
        trades: pd.DataFrame,
        start_date,
        end_date
        # account_market_prices: pd.DataFrame
    ) -> List[Dict]:

        # print(f"_calculate_daily_timeline() in progress")
        # time5 = datetime.now()

        """Calculate holdings for every day (with carry-forward)."""
        timeline = []
        holdings_state = {}  # symbol -> bucket -> {qty, avg_price, invested_value}
        
        # Create date range for all days
        current_date = start_date
        
        while current_date <= end_date:
            # Get trades for this date
            date_trades = trades[trades['trade_date'] == current_date]
            
            if not date_trades.empty:
                # Process trades and update holdings
                trades_count_today = self._process_trades_for_date(
                    date_trades, holdings_state
                )
            else:
                # No trades today - carry forward
                trades_count_today = {}
                for symbol in holdings_state:
                    for bucket in holdings_state[symbol]:
                        key = f"{symbol}:{bucket}"
                        trades_count_today[key] = 0
            
            # Create snapshot for this date
            timeline_entry = self._create_timeline_entry(
                current_date, holdings_state, trades_count_today
            )
            timeline.append(timeline_entry)
            
            # Move to next day
            current_date += timedelta(days=1)

        # print(f"Time taken: {datetime.now() - time5 }")
        return timeline
    
    def _process_trades_for_date(
        self,
        date_trades: pd.DataFrame,
        holdings_state: Dict
    ) -> Dict[str, int]:
        
        # print(f"_process_trades_for_date() in progress")
        # time6 = datetime.now()

        """
        Process all trades for a date and update holdings state.
        
        Args:
            date_trades: Trades for this specific date
            holdings_state: Mutable dict tracking current holdings by symbol and bucket
        
        Returns:
            Dict mapping "symbol:bucket" -> number of trades today
        """
        trades_count = {}
        
        # Process each trade individually to track bucket
        for _, trade in date_trades.iterrows():
            symbol = trade['tradingsymbol']
            tag = trade['tag']
            key = f"{symbol}:{tag}"
            
            trades_count[key] = trades_count.get(key, 0) + 1
            
            self._apply_trade(
                symbol,
                tag,
                trade['transaction_type'],
                trade['quantity'],
                trade['average_price'],
                holdings_state
            )
        # print(f"Time taken: {datetime.now() - time6 }")
        return trades_count
    
    def _apply_trade(
        self,
        symbol: str,
        tag: str,
        transaction_type: str,
        quantity: int,
        price: float,
        holdings_state: Dict
    ):

        """
        Apply a single trade to update holdings state.
        
        Updates holdings_state in-place with bucket-aware tracking.
        Structure: holdings_state[symbol][bucket] = {quantity, average_price, invested_value}
        """
        if symbol not in holdings_state:
            holdings_state[symbol] = {}
        
        if tag not in holdings_state[symbol]:
            holdings_state[symbol][tag] = {
                'quantity': 0,
                'average_price': 0.0,
                'invested_value': 0.0
            }
        
        current = holdings_state[symbol][tag]
        
        if transaction_type == 'BUY':
            # Calculate new weighted average price
            old_value = current['quantity'] * current['average_price']
            new_value = quantity * price
            total_quantity = current['quantity'] + quantity
            
            if total_quantity > 0:
                new_avg_price = (old_value + new_value) / total_quantity
            else:
                new_avg_price = price
            
            # Update holdings
            current['quantity'] = total_quantity
            current['average_price'] = round(new_avg_price, 2)
            current['invested_value'] = round(
                current['quantity'] * current['average_price'], 2
            )
        
        elif transaction_type == 'SELL':
            # Reduce quantity, keep average price same
            current['quantity'] -= quantity
            
            if current['quantity'] < 0:
                # Handle short positions or errors
                pass
            
            # Recalculate invested value with same avg price
            current['invested_value'] = round(
                current['quantity'] * current['average_price'], 2
            )
            
            # If completely sold out, remove this bucket
            if current['quantity'] == 0:
                del holdings_state[symbol][tag]
                # If no buckets left for symbol, remove symbol
                if not holdings_state[symbol]:
                    del holdings_state[symbol]
    
    def _create_timeline_entry(
        self,
        date,
        holdings_state: Dict,
        trades_count_today: Dict[str, int]
        # account_market_prices: pd.DataFrame
    ) -> Dict:
        """Create a timeline entry for a specific date with bucket breakdown."""
        
        # print(f"_create_timeline_entry() in progress")
        # time7 = datetime.now()


        ##########################
        # time_p1 = datetime.now()
        ##########################
        # Prepare bucket-wise holdings
        buckets = {}

        # price_map = account_market_prices.set_index(['tradingsymbol', 'exchange_token', 'date'])['close'].to_dict()
        

        ##########################
        # print(f"part1: {datetime.now() - time_p1 }")
        ##########################
        ##########################
        # time_p2 = datetime.now()
        ##########################

        # sec_id_dict = account_market_prices.set_index(['tradingsymbol'])['exchange_token'].to_dict()

        ##########################
        # print(f"part2: {datetime.now() - time_p2 }")
        ##########################

        def get_mkt_close_price(symbol, date_str):

            try:
                sec_id = self.sec_id_dict.get(symbol)
                close_price = self.price_map.get((symbol, str(sec_id), date_str))
                
                while close_price is None:
                    date_str = date_str - timedelta(days = 1)
                    close_price = self.price_map.get((symbol, str(sec_id), date_str))
                return close_price
            
            except Exception as e:
                print(symbol)
                print(date_str)
                print(e)
                return None
            
            

        # First, organize by bucket
        for symbol, bucket_holdings in holdings_state.items():
            for bucket, holding in bucket_holdings.items():
                if holding['quantity'] <= 0:
                    continue
                    
                if bucket not in buckets:
                    buckets[bucket] = {
                        'invested_value': 0.0,
                        'market_value': 0.0,
                        'unique_symbols': 0,
                        'holdings': []
                    }
                mkt_close = get_mkt_close_price(symbol=symbol, date_str=date)
                if mkt_close == None:
                    mkt_val = None
                else:
                    mkt_val = mkt_close * holding['quantity']
                # Create holding entry for this bucket
                holding_entry = {
                    'symbol': symbol,
                    'quantity': holding['quantity'],
                    'average_price': holding['average_price'],
                    'invested_value': holding['invested_value'],
                    'trades_today': trades_count_today.get(f"{symbol}:{bucket}", 0),
                    'market_price': mkt_close,
                    'market_value': mkt_val,
                    'unrealized_pnl': round(mkt_val - holding['invested_value'], 2) if mkt_val is not None else None  # CALCULATE PNL
                }
                
                buckets[bucket]['holdings'].append(holding_entry)
                buckets[bucket]['invested_value'] += holding['invested_value']
                # buckets[bucket]['market_value'] += holding['market_value']
                if mkt_val is not None:
                    buckets[bucket]['market_value'] += mkt_val
        

        # Calculate unique symbols per bucket and sort holdings
        for bucket in buckets:
            buckets[bucket]['unique_symbols'] = len(buckets[bucket]['holdings'])
            buckets[bucket]['holdings'].sort(key=lambda x: x['symbol'])
            buckets[bucket]['invested_value'] = round(buckets[bucket]['invested_value'], 2)
            buckets[bucket]['market_value'] = round(buckets[bucket]['market_value'], 2)  # ROUND IT
        
        # Create aggregated flat holdings list (across all buckets)
        aggregated_holdings = {}
        total_invested = 0.0
        total_market_value = 0.0
        total_quantity = 0


        for symbol, bucket_holdings in holdings_state.items():
            for bucket, holding in bucket_holdings.items():
                if holding['quantity'] <= 0:
                    continue
                
                if symbol not in aggregated_holdings:
                    aggregated_holdings[symbol] = {
                        'quantity': 0,
                        'total_value': 0.0,
                        'total_market_value': 0.0,
                        'trades_today': 0
                    }
                
                aggregated_holdings[symbol]['quantity'] += holding['quantity']
                aggregated_holdings[symbol]['total_value'] += holding['invested_value']
                aggregated_holdings[symbol]['total_market_value'] += mkt_val
                aggregated_holdings[symbol]['trades_today'] += trades_count_today.get(
                    f"{symbol}:{bucket}", 0
                )


        # Convert aggregated holdings to list format
        holdings_list = []
        for symbol, agg in aggregated_holdings.items():
            avg_price = agg['total_value'] / agg['quantity'] if agg['quantity'] > 0 else 0
            mkt_close = get_mkt_close_price(symbol=symbol, date_str=date)
            if mkt_close == None:
                mkt_val = None
            else:
                mkt_val = mkt_close * agg['quantity']
            holding_entry = {
                'symbol': symbol,
                'quantity': agg['quantity'],
                'average_price': round(avg_price, 2),
                'invested_value': round(agg['total_value'], 2),
                'trades_today': agg['trades_today'],
                'market_price': mkt_close,
                'market_value': mkt_val,
                'unrealized_pnl': round(mkt_val - agg['total_value'], 2) if mkt_val is not None else None  
            }
            holdings_list.append(holding_entry)
            total_invested += agg['total_value']
            
            if mkt_val is not None:
                total_market_value += mkt_val
            
            total_quantity += agg['quantity']
        
        # Sort by symbol
        holdings_list.sort(key=lambda x: x['symbol'])

        total_unrealized_pnl = round(total_market_value - total_invested, 2) if total_market_value > 0 else None

        # print(f"Time taken (create_timeline_entry): {datetime.now() - time7 }")
        
        return {
            'date': date.isoformat(),
            'holdings': holdings_list,
            'summary': {
                'total_invested_value': round(total_invested, 2),
                'total_market_value': round(total_market_value, 2) if total_market_value > 0 else None, 
                'total_unrealized_pnl': total_unrealized_pnl,
                'unique_symbols': len(holdings_list),
                'total_quantity_all_symbols': total_quantity
            },
            'buckets': buckets
        }
    
    def _empty_response(self, account_id: str, granularity: str) -> Dict:
        """Return empty response when no trades found."""
        return {
            'account_id': account_id,
            'currency': 'INR',
            'granularity': granularity,
            'date_range': {
                'start_date': None,
                'end_date': None,
                'total_days': 0,
                'trading_days': 0
            },
            'timeline': []
        }
    

In [15]:
def load_trades_data():
    """Load trades data from your database/source"""
    # Replace with your actual data loading logic
    # Example: df = pd.read_sql("SELECT * FROM trades", conn)
    # For now, assuming you have this function
    # from your_data_module import get_trades_dataframe

    trades_df = sqlite_db.get_data("select * from users_auth_trades")
    trades_df['instrument_token'] = pd.to_numeric(trades_df['instrument_token'], errors='coerce')
    qdb_df = quest_db.read_questdb_dataframe(query= "select * from kite_instruments_list")

    merged = trades_df.merge(qdb_df[['instrument_token', 'exchange_token']], 
                left_on='instrument_token',
                right_on='instrument_token',
                how='left')


    merged['tag'] = merged.apply(lambda x: 'etf' if x['tradingsymbol'] in ['GOLDBEES', 'GOLDBEES-E', 'JUNIORBEES', 'LIQUIDCASE', 'NIFTYBEES'] else None, 
                                    axis = 1)
    

    return merged

def get_portfolio_timeline():
    """Get calculated portfolio timeline"""
    trades_df = load_trades_data()
    time0 = datetime.now()
    calculator = HoldingsTimelineCalculator(trades_df)
    portfolio = calculator.calculate_timeline(
        account_id='AGL883',
        granularity='daily'
    )
    print(f"Total time taken: {datetime.now()-time0}")
    return portfolio

In [16]:
pf = get_portfolio_timeline()

# original
# part1: 0:00:00.033836
# part2: 0:00:00.026350
# Time taken (create_timeline_entry): 0:00:00.060494



Total time taken: 0:00:00.911727


In [10]:
trades_df = load_trades_data()

In [12]:
# merged_dat.head(5)
pf

{'account_id': 'AGL883',
 'currency': 'INR',
 'granularity': 'daily',
 'date_range': {'start_date': '2022-11-09',
  'end_date': '2026-01-16',
  'total_days': 1165,
  'trading_days': 36},
 'timeline': [{'date': '2022-11-09',
   'holdings': [{'symbol': 'JUNIORBEES',
     'quantity': 1,
     'average_price': 460.19,
     'invested_value': 460.19,
     'trades_today': 1,
     'market_price': 453.02,
     'market_value': 453.02,
     'unrealized_pnl': -7.17},
    {'symbol': 'NIFTYBEES',
     'quantity': 5,
     'average_price': 198.83,
     'invested_value': 994.15,
     'trades_today': 1,
     'market_price': 196.85,
     'market_value': 984.25,
     'unrealized_pnl': -9.9}],
   'summary': {'total_invested_value': 1454.34,
    'total_market_value': 1437.27,
    'total_unrealized_pnl': -17.07,
    'unique_symbols': 2,
    'total_quantity_all_symbols': 6},
   'buckets': {'etf': {'invested_value': 1454.34,
     'market_value': 1437.27,
     'unique_symbols': 2,
     'holdings': [{'symbol': 'J